In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the trained xgb_model to make predictions on X_test_cleaned
y_pred_xgb = xgb_model.predict(X_test_cleaned)

# Calculate evaluation metrics
accuracy_xgb = accuracy_score(y_test_true_rf, y_pred_xgb)
precision_xgb = precision_score(y_test_true_rf, y_pred_xgb, average='weighted')
recall_xgb = recall_score(y_test_true_rf, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test_true_rf, y_pred_xgb, average='weighted')

# Print the calculated metrics
print(f"XGBClassifier Performance on Test Data:")
print(f"  Accuracy: {accuracy_xgb:.4f}")
print(f"  Precision (weighted): {precision_xgb:.4f}")
print(f"  Recall (weighted): {recall_xgb:.4f}")
print(f"  F1-Score (weighted): {f1_xgb:.4f}")

In [ ]:
import numpy as np

# 1. Create y_train_binary by mapping original categories 0, 1, 2, 3 to 0 and category 4 to 1
y_train_binary = np.where(y_train_flat == 4, 1, 0)

# 2. Create y_test_binary by applying the same mapping to y_test_flat
y_test_binary = np.where(y_test_flat == 4, 1, 0)

# 3. Print the unique values and their counts for both y_train_binary and y_test_binary
print("Unique values and counts for y_train_binary:")
unique_train, counts_train = np.unique(y_train_binary, return_counts=True)
for val, count in zip(unique_train, counts_train):
    print(f"  Class {val}: {count} instances")

print("\nUnique values and counts for y_test_binary:")
unique_test, counts_test = np.unique(y_test_binary, return_counts=True)
for val, count in zip(unique_test, counts_test):
    print(f"  Class {val}: {count} instances")

# 4. Print the shapes of y_train_binary and y_test_binary
print(f"\nShape of y_train_binary: {y_train_binary.shape}")
print(f"Shape of y_test_binary: {y_test_binary.shape}")

In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

# 1. y_train_binary and y_test_binary are already 1D NumPy arrays from the previous step

# 2. Determine the number of unique classes from y_train_binary to configure the model output
n_classes_binary = len(np.unique(y_train_binary))
print(f"Number of classes for binary classification inferred: {n_classes_binary}")

# 3. Set the device to 'cuda' if a GPU is available, otherwise 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 4. Instantiate a TabNetClassifier object named model_classifier_binary
model_classifier_binary = TabNetClassifier(
    verbose=10,
    seed=42, # For reproducibility
    device_name=device,
    n_d=64, # Dimension of the decision prediction layer
    n_a=64, # Dimension of the attention embedding for each mask
    n_steps=5, # Number of sequential attention-decision steps
    gamma=1.5, # Multiplier for feature importance of successive attention masks
    cat_emb_dim=1 # For categorical features, which are not present after one-hot encoding in this case
)

# 5. Train the model_classifier_binary using the .fit() method
model_classifier_binary.fit(
    X_train=X_train_scaled_np, y_train=y_train_binary,
    eval_set=[(X_test_scaled_np, y_test_binary)],
    eval_name=['val'],
    eval_metric=['accuracy'], # Accuracy for binary classification evaluation
    max_epochs=100,
    patience=100, # Stop training if validation accuracy does not improve for 1000 consecutive epochs
    batch_size=1024, # Batch size for training
    virtual_batch_size=128, # Virtual batch size for ghost batch normalization
    num_workers=0 # Number of workers for data loading (0 means main process)
)

print("Binary TabNetClassifier model training complete.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Extract the 'loss' values from model_classifier_binary.history for training loss.
# 2. Extract the 'val_accuracy' values from model_classifier_binary.history for validation accuracy.
epochs = range(len(model_classifier_binary.history['loss']))
training_loss = model_classifier_binary.history['loss']
validation_accuracy = model_classifier_binary.history['val_accuracy']

# 3. Create a figure and a primary Y-axis for plotting the training loss.
fig, ax1 = plt.subplots(figsize=(10, 6))

# 4. Plot the training loss on the primary Y-axis, labeling it 'Training Loss', and set its color.
color = 'tab:red'
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss', color=color)
ax1.plot(epochs, training_loss, color=color, label='Training Loss')
ax1.tick_params(axis='y', labelcolor=color)

# 5. Create a secondary Y-axis that shares the same X-axis as the primary Y-axis.
ax2 = ax1.twinx()

# 6. Plot the extracted validation accuracy on this secondary Y-axis, labeling it 'Validation Accuracy', and set a different color for the plot.
color = 'tab:blue'
ax2.set_ylabel('Validation Accuracy', color=color)
ax2.plot(epochs, validation_accuracy, color=color, label='Validation Accuracy')
ax2.tick_params(axis='y', labelcolor=color)

# 7. Add a title to the plot.
plt.title('Binary TabNetClassifier: Training Loss and Validation Accuracy Over Epochs')

# 8. Adjust layout to prevent labels from overlapping
fig.tight_layout()

# 9. Manually combine the legends from both axes and display them on the plot.
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

# 10. Display the plot.
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. Predict probabilities for the positive class (class 1) on the X_test_scaled_np
# For binary classification, predict_proba returns probabilities for both classes [P(class 0), P(class 1)]
y_pred_proba_binary = model_classifier_binary.predict_proba(X_test_scaled_np)[:, 1] # Get probabilities for the positive class (class 1)

# 2. Calculate the False Positive Rate (FPR) and True Positive Rate (TPR)
fpr_binary, tpr_binary, _ = roc_curve(y_test_binary, y_pred_proba_binary)

# 3. Calculate the Area Under the Curve (AUC)
roc_auc_binary = auc(fpr_binary, tpr_binary)

# 4. Create a plot for the ROC curve
plt.figure(figsize=(8, 6))

# 5. Plot the ROC curve
plt.plot(fpr_binary, tpr_binary, color='darkorange', lw=2,
         label=f'ROC curve (area = {roc_auc_binary:0.2f})')

# 6. Plot a diagonal dashed line as a 'Random classifier' baseline
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier (area = 0.50)')

# 7. Add appropriate labels for the x-axis and y-axis
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

# 8. Set the title of the plot
plt.title('Receiver Operating Characteristic (ROC) Curve for Binary Classification')

# 9. Add a legend to the plot
plt.legend(loc="lower right")
plt.grid(True)

# 10. Display the plot
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Predict class labels for the X_test_scaled_np dataset
y_pred_classes_binary = model_classifier_binary.predict(X_test_scaled_np)

# 2. Compute the confusion matrix
cm_binary = confusion_matrix(y_test_binary, y_pred_classes_binary)

# 3. Display the confusion matrix as a normalized plot
fig, ax = plt.subplots(figsize=(8, 8))
display = ConfusionMatrixDisplay.from_predictions(
    y_true=y_test_binary,
    y_pred=y_pred_classes_binary,
    cmap='Blues',
    normalize=None,
    ax=ax
)

# 4. Add a title to the plot
ax.set_title('Normalized Confusion Matrix for Binary TabNetClassifier')

# 5. Display the plot
plt.show()